# SPOD to Mapping Excel
- Prerequisites: 
  - Anaconda packages: `xlsxwriter` pandas, openpyxl, seaborn`


## Result

Excel sheet containing:

Sheet with all Datapoints (Systems, Tables and Columns) mapped against the Information Model (Entity, Attribute)
Table covering the overview sheet

Analog dev_x_mapping

Optional:
Sheet per System - IM containing sample data

## Structure

1. Define mapping between SPOD (json) and columns in the resulting Excel sheet

## Configuration
The following parameters has to be definded when running as regular python script

In [1]:
LIBRARY = '../../pythonWork/pythonSource'

DESTINATION = 'mapping.xlsx'

MODEL_SOURCE = '/Users/bue/projects/geberit/DEAP/DB/IM_GEBERIT.json'
MODEL_SOURCE = '/Users/bue/projects/sika/Sika-IM/DB/Sika-IM.json'
MODEL_SOURCE = '/Users/bue/projects/sika/Sika-IM/DB/Sika-IM.import.regen.json'
MODEL_SOURCE = '/Users/bue/projects/sika/Sika-IM/DB/Sika-IM.import.json'

MODEL_SOURCE = '/Users/bue/projects/sika/Sika-IM/DB/Sika-IM_FR.json'
MODEL_SOURCE = '/Users/bue/projects/sika/Sika-IM/DB/reload.json'

MODEL_SOURCE = '/Users/bue/projects/sika/Sika-IM/DB/Sika-IM.json'
MODEL_SOURCE = '/Users/bue/projects/sika/Sika-IM/DB/Sika-IM.regen.json'

## Check prerequisites

In [2]:
import sys
import logging
import os
import json
import yaml
from pathlib import Path

In [3]:
import matplotlib.colors as mcolors
import seaborn as sns

In [4]:
# openpyxl
from openpyxl import Workbook
from openpyxl.worksheet.table import Table
from openpyxl.utils.cell import get_column_letter
from openpyxl.styles import PatternFill

In [5]:
spod_file = Path(MODEL_SOURCE)
assert spod_file.is_file(), f"Cannot find SPOD file '{spod_file.resolve()}'"

with open(spod_file, 'r') as src:
    spod = json.load(src)
assert spod['model'] is not None
print(f"Loaded SPOD containing {spod['model']} from '{spod_file.resolve()}'")

Loaded SPOD containing {'name': 'Sika-IM', 'type': 'logical', 'language': 'de', 'uc': 'stb', 'dc': '2021-10-26 12:05:29 UTC', 'um': 'SPOD', 'dm': '2022-05-18 18:03:35.940741'} from '/Users/bue/projects/sika/Sika-IM/DB/Sika-IM.regen.json'


In [6]:
print(f"Loaded {spod_file.resolve()}\n{spod['model']}\nVersion {spod['_imprint_']}")
print(f"Languages: {list(spod['languages'].keys())}")
mapdict = {}
for entry in ['entities', 'attributes', 'systems', 'columns']:
    print(f"- {entry}: {len(spod[entry])}")

Loaded /Users/bue/projects/sika/Sika-IM/DB/Sika-IM.regen.json
{'name': 'Sika-IM', 'type': 'logical', 'language': 'de', 'uc': 'stb', 'dc': '2021-10-26 12:05:29 UTC', 'um': 'SPOD', 'dm': '2022-05-18 18:03:35.940741'}
Version {'database': '/Users/bue/projects/sika/Sika-IM/DB/Sika-IM.db', 'created': '2022-05-27 11:07:56.513804', 'Modelversion': '1.9', 'hashvalue': 2836600587353357128, 'git-revision': 'a7a7381', 'comment': 'Entries ending with + represent denormalized data and are not checked for consistency while reading back'}
Languages: ['de', 'en', 'fr']
- entities: 89
- attributes: 137
- systems: 35
- columns: 942


## Initialize logging

In [7]:
import logging
from logging import handlers
from datetime import datetime

stamp = datetime.now()
run_stamp = stamp.strftime("%Y-%m-%d-%H-%M-%S")

os.makedirs('log', exist_ok=True)
logfile = f'log/sharepoint-list-sync-{run_stamp}.log'

handler = handlers.RotatingFileHandler(logfile, maxBytes=(1024 * 1024 * 10), backupCount=10)
handler.setLevel(logging.DEBUG)

formatter = logging.Formatter("%(asctime)s [%(threadName)s] - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)

console_log_handler = logging.StreamHandler()
console_formatter = logging.Formatter("%(levelname)s - %(message)s")
console_log_handler.setFormatter(console_formatter)
console_log_handler.setLevel(logging.INFO)

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
logger.addHandler(handler)
logger.addHandler(console_log_handler)

urlliblogger = logging.getLogger('urllib3.connectionpool')
urlliblogger.setLevel(logging.DEBUG)

## Use the fyayc SPOD library

In [8]:
toolpath = Path(LIBRARY)
assert toolpath.is_dir(), f"{toolpath.reslove()} is not a directory. The constant 'LIBRARY' must point to the library. Default = 'pythonWork/pythonSource'."
sys.path.insert(0, str(toolpath))

from PUBLISH_MODEL.excel.mapping_publisher import generate
from SSOT_infra.translator import Translator

## Translation shortcut tr

In [9]:
translator = Translator('de')

## List structure definition

In [10]:
columns_mapped = {}
system_index = {}

In [11]:
headings_im = ["FQN", "EID", "AID", "Entity (EN) ", "Attribute (EN)", "Entity (DE) ", "Attribute (DE)", "Entity (FR) ", "Attribute (FR)", "Examples", "Description", "#"]

In [12]:
len(headings_im)

12

In [13]:
def emit_system_columns(spod: dict) -> [str]:
    result = []
    for key, system in spod['systems'].items():
        system_index[key] = len(headings_im) + len(result)
#        result.append(key + ':FQN')
        result.append(system['name'])
#        result.append(key + ':REF')
    return result

In [14]:
systems = emit_system_columns(spod)

In [15]:
systems

['ADDOK_MATCO / COFAQ',
 'Amazon FR',
 'Bigmat_SIKA',
 'Bricoman',
 'CMEM_SIKA',
 'CMEM_SIKA_200421',
 'COFAQ_sika_france-mp-brut_net',
 'CXM Access',
 'CXM DPB Schnittstelle',
 'CXM ProductUp FR',
 'Chausson_Tarifs SIKA',
 'Excel ADDOK_MATCO:01.02.2022 Tabelle: Correspondance Nom douaniere',
 'Excel ADDOK_MATCO:01.02.2022 Tabelle: Matrice Tabelle: Examples',
 'Excel ADDOK_MATCO:01.02.2022 Tabelle: Notice format de matrice',
 'Excel ADDOK_MATCO:01.02.2022 Tabelle: Tables unités Tabelle Correspondance UB Tabelle: Correspondance QCT',
 'FAB-DIS',
 'GEDIMAT_Matrice_ARTICLE-SIKA',
 'GEDIMAT_TARIF',
 'Jedele',
 'LA PDB_Création',
 'MEG',
 'MR BRICOLAGE',
 'MegaSTAMM',
 'POINTE P_SIKA_FRANCE',
 'PROLIANS_348-SIKA FRANCE',
 'SAP Export - EAN doublon et code sap remplacant',
 'SAP Export - fichier unique données sap',
 'SAP-ERP',
 'Sika standard',
 'Standart Abmessungen',
 'Standart KD',
 'Standart Link',
 'VFG',
 'Wertschöpfer',
 'free']

In [16]:
def export_column(key: str, column: dict) -> []:
    result = [
                #column['interface-id+'] + '.' + column['table-id'] + '.' + key,
                column['name'],
                #column['interface_col_id'],
            ]
    return result
    
def firsthit(spod: dict, attribute_key: str, system_key: str) -> []:
    for ckey, column in spod['columns'].items():
        if ckey not in columns_mapped and system_key == column['interface-id+'] and attribute_key in column['attributesmapped']:
            result = export_column(ckey, column)
            columns_mapped[ckey] = attribute_key
            return result
    #return [ '', '', '' ]
    return [ '', ]

In [17]:
def emit_row(spod: dict, attribute_key: str, attribute: dict, translator: Translator) -> []:
    enti_key = attribute['entity']
    entity = spod['entities'].get(enti_key)
    assert entity is not None, f"Missing entity {enti_key}"
    result = [
        enti_key + ':' + attribute_key,
        enti_key,
        attribute_key,
        translator.tr(entity['name'], 'en'),
        translator.tr(attribute['name'], 'en'),
        translator.tr(entity['name'], 'de'),
        translator.tr(attribute['name'], 'de'),
        translator.tr(entity['name'], 'fr'),
        translator.tr(attribute['name'], 'fr'),
        ', '.join(translator.tr(attribute.get('examples'), 'en')),
        translator.tr(attribute['descr'], 'en'),
        len(attribute['columnsmapped+']),
    ]

    for skey in spod['systems'].keys():
        mapping = firsthit(spod, attribute_key, skey)
        result = result + mapping

    return result

In [18]:
headings = headings_im + systems
f"Columns ({len(headings)}): {', '.join(headings)}"

'Columns (47): FQN, EID, AID, Entity (EN) , Attribute (EN), Entity (DE) , Attribute (DE), Entity (FR) , Attribute (FR), Examples, Description, #, ADDOK_MATCO / COFAQ, Amazon FR, Bigmat_SIKA, Bricoman, CMEM_SIKA, CMEM_SIKA_200421, COFAQ_sika_france-mp-brut_net, CXM Access, CXM DPB Schnittstelle, CXM ProductUp FR, Chausson_Tarifs SIKA, Excel ADDOK_MATCO:01.02.2022 Tabelle: Correspondance Nom douaniere, Excel ADDOK_MATCO:01.02.2022 Tabelle: Matrice Tabelle: Examples, Excel ADDOK_MATCO:01.02.2022 Tabelle: Notice format de matrice, Excel ADDOK_MATCO:01.02.2022 Tabelle: Tables unités Tabelle Correspondance UB Tabelle: Correspondance QCT, FAB-DIS, GEDIMAT_Matrice_ARTICLE-SIKA, GEDIMAT_TARIF, Jedele, LA PDB_Création, MEG, MR BRICOLAGE, MegaSTAMM, POINTE P_SIKA_FRANCE, PROLIANS_348-SIKA FRANCE, SAP Export - EAN doublon et code sap remplacant, SAP Export - fichier unique données sap, SAP-ERP, Sika standard, Standart Abmessungen, Standart KD, Standart Link, VFG, Wertschöpfer, free'

# Create data table (content)

In [19]:
data_table = [ emit_row(spod, key, attribute, translator) for key, attribute in spod['attributes'].items() ]

In [20]:
len(data_table)

137

In [21]:
f"Already mapped {len(columns_mapped)} columns of {len(spod['columns'])}"

'Already mapped 171 columns of 942'

In [22]:
list(columns_mapped.items())[0:3]

[('COLU570', 'ATTR109'), ('COLU768', 'ATTR111'), ('COLU677', 'ATTR123')]

## Append unmapped columns to the bottom

In [23]:
def aux_row(spod: dict, key: str, column: dict, translator: Translator) -> []:
    mapped = column['attributesmapped']
    if len(mapped) > 0:
        attrkey = mapped[0]
        attr = spod['attributes'][attrkey]
        result = emit_row(spod, attrkey, attr, translator)
        result.extend( [ None ] * (len(headings) - len(result))  )
    else:
        result = [ None ] * len(headings)
    
    map_count = len(column['attributesmapped'])
    result[len(headings_im) - 1] = map_count

    index = system_index[column['interface-id+']]
    values = export_column(key, column)
    result[index + 0] = values[0]
#    result[index + 1] = values[1]
#    result[index + 2] = values[2]
    
    
    return result

In [24]:
remainder = [ aux_row(spod, key, spod['columns'][key], translator) for key in filter(lambda key: key not in columns_mapped.keys(), spod['columns'].keys()) ]

In [25]:
data_table = data_table + remainder

In [26]:
### Sort by Attribute FQN
data_table.sort(key=lambda r: r[0] if r[0] is not None else '\uFFFF')

# Prepare Excel Workbook

In [27]:
wb = Workbook()
ws = wb.active
ws.title = 'Mapping'

# add column headings. NB. these must be strings
ws.append(headings)
for row in data_table:
    ws.append(row)

## Define a data table readable by Sharepoint

In [28]:
tab = Table(displayName="Mapping", ref=f"A1:{get_column_letter(len(headings))}{len(data_table)+1}")
ws.add_table(tab)
tab._initialise_columns()

for column, value in zip(tab.tableColumns, headings):
    column.name = value

## Styling

In [29]:
pal = list(sns.color_palette('pastel'))

for column_index in range(len(headings_im), len(headings)):
    color_index = int((column_index - len(headings_im)) / 3)
    color = pal[color_index % len(pal)]
    rgb = str(mcolors.to_hex(color))[1:]
    #print(rgb)
    for cell in ws[get_column_letter(column_index + 1)]:
        cell.fill = PatternFill(fgColor=rgb, fill_type = "solid")

### Resize and hide columns

In [30]:
ws.column_dimensions['B'].hidden= True
ws.column_dimensions['C'].hidden= True

ws.column_dimensions['F'].hidden= True
ws.column_dimensions['G'].hidden= True
ws.column_dimensions['H'].hidden= True
ws.column_dimensions['I'].hidden= True


In [31]:
base = len(headings_im)
index = 0
for system in spod['systems'].values():
    colnr = base + 1 + (index * 3)
    col_letter = get_column_letter(colnr)
    dim = ws.column_dimensions[col_letter]
    dim.hidden= True
    
    col_letter_cid = get_column_letter(colnr + 2)
    dim = ws.column_dimensions[col_letter_cid]
    dim.hidden= True
    
    print(f"Hiding columns {col_letter} ({ws[col_letter + '1'].value})"
          f" and {col_letter_cid} ({ws[col_letter_cid + '1'].value}) on System {system['name']} idx {colnr}")
    
    index += 1

Hiding columns M (ADDOK_MATCO / COFAQ) and O (Bigmat_SIKA) on System ADDOK_MATCO / COFAQ idx 13
Hiding columns P (Bricoman) and R (CMEM_SIKA_200421) on System Amazon FR idx 16
Hiding columns S (COFAQ_sika_france-mp-brut_net) and U (CXM DPB Schnittstelle) on System Bigmat_SIKA idx 19
Hiding columns V (CXM ProductUp FR) and X (Excel ADDOK_MATCO:01.02.2022 Tabelle: Correspondance Nom douaniere) on System Bricoman idx 22
Hiding columns Y (Excel ADDOK_MATCO:01.02.2022 Tabelle: Matrice Tabelle: Examples) and AA (Excel ADDOK_MATCO:01.02.2022 Tabelle: Tables unités Tabelle Correspondance UB Tabelle: Correspondance QCT) on System CMEM_SIKA idx 25
Hiding columns AB (FAB-DIS) and AD (GEDIMAT_TARIF) on System CMEM_SIKA_200421 idx 28
Hiding columns AE (Jedele) and AG (MEG) on System COFAQ_sika_france-mp-brut_net idx 31
Hiding columns AH (MR BRICOLAGE) and AJ (POINTE P_SIKA_FRANCE) on System CXM Access idx 34
Hiding columns AK (PROLIANS_348-SIKA FRANCE) and AM (SAP Export - fichier unique données sa

## Save to Excel file

In [32]:
dest = Path(DESTINATION)
wb.save(dest)
print(f"Wrote {dest.resolve()}")

Wrote /Users/bue/dev/fyyccim-tools-master/notebooks/mig/mapping.xlsx


# xlsxwriter Approach

In [33]:
import xlsxwriter

In [34]:
xlsx_destination = 'IM_' + dest.name
workbook = xlsxwriter.Workbook(xlsx_destination)

title_format = workbook.add_format({'bold': True, 'font_color': 'black', 'font_size': 20})
column_head_format = workbook.add_format({'bold': True, 'bg_color': '#A0A0A0'})



In [35]:
## Summary is first sheet, but will be filled last
summary = workbook.add_worksheet('Summary')

In [36]:
## Prepare Mapping sheet

In [37]:
worksheet = workbook.add_worksheet('Mapping')

col = 0
for header in headings:
    worksheet.write(0, col, header)
    col += 1

row = 1
for entry in data_table:
    col = 0
    for item in entry:
        worksheet.write(row, col, item)
        col += 1
    row += 1

### Define Table

In [38]:
table_column_headers = [ { 'header': name } for name in headings ]

In [39]:
worksheet.add_table(0, 0, len(data_table) + 1, len(headings) - 1, { 
    'name': 'mapping',
    'banded_rows': True,
    'columns': table_column_headers,
})

0

### Styling

In [40]:
# FQN width
worksheet.set_column(0, 0, 20)

# Hide EID, AID on the left
worksheet.set_column(1, 3, 10, None, { 'hidden': 1, })

# EID, AID
worksheet.set_column(3, 5, 20)

# Hide attribute name translations (DE, FR)
worksheet.set_column(5, 8, 40, None, { 'hidden': 1, })

base = len(headings_im)
index = 0
for system in spod['systems'].values():
#    colnr = base + (index * 3)
    colnr = base + index
#    worksheet.set_column(colnr, colnr, None, None, { 'hidden': 1, })
    
    # Column name on system
    worksheet.set_column(colnr + 1, colnr + 1, 30)
    
    # Technical reference
 #   worksheet.set_column(colnr + 2, colnr + 2, None, None, { 'hidden': 1, })        
    index += 1

## Add one sheet per system

In [41]:
def fill_worksheet(spod: dict, skey: str, system: str, sheet):
    row = 0
    sheet.write(row, 0, 'Table Key', column_head_format)
    sheet.write(row, 1, 'Table Name', column_head_format)
    sheet.write(row, 2, 'Column Key', column_head_format)
    sheet.write(row, 3, 'Tech-ID', column_head_format)
    sheet.write(row, 4, 'Name', column_head_format)
    sheet.write(row, 5, 'Description', column_head_format)
    sheet.write(row, 6, '|', column_head_format)
    sheet.write(row, 7, 'IM Attributes', column_head_format)
    
    row += 1
    columns = sorted(list(spod['columns'].items()), key=lambda c: c[1]['table-name+'])
    for ckey, column in columns:
        if column['interface-id+'] == skey:
            sheet.write(row, 0, column['table-id'])
            sheet.write(row, 1, column['table-name+'])
            sheet.write(row, 2, ckey)
            sheet.write(row, 3, column['interface_col_id'])
            sheet.write(row, 4, column['name'])
            sheet.write(row, 5, translator.tr(column['descr'], 'de'))
            sheet.write(row, 6, '|')
            sheet.write(row, 7, ', '.join(column['attributesmapped']))
            row += 1
    
    return row

In [42]:
import re

worksheets = dict()

for key, system in spod['systems'].items():
    title = re.sub(r'[\:\[\]*?/\\]', '_', system['name'])
    length = min(25, len(title))
    t = key.replace('INTF','') + ' ' + title[:length]
    if t.lower() in worksheets.keys():
        t = key.replace('INTF','') + ' ' + title[max(0, len(title) - 25):]
    worksheet = workbook.add_worksheet(t)
    worksheets[t.lower()] = worksheet
    rows = fill_worksheet(spod, key, system, worksheet)
    print(f"{key}: {system['name']} -> {t} with {rows} rows")

INTF4594: ADDOK_MATCO / COFAQ -> 4594 ADDOK_MATCO _ COFAQ with 74 rows
INTF1191: Amazon FR -> 1191 Amazon FR with 2 rows
INTF2184: Bigmat_SIKA -> 2184 Bigmat_SIKA with 1 rows
INTF5421: Bricoman -> 5421 Bricoman with 1 rows
INTF4595: CMEM_SIKA -> 4595 CMEM_SIKA with 267 rows
INTF2185: CMEM_SIKA_200421 -> 2185 CMEM_SIKA_200421 with 47 rows
INTF2186: COFAQ_sika_france-mp-brut_net -> 2186 COFAQ_sika_france-mp-brut with 1 rows
INTF552: CXM Access -> 552 CXM Access with 97 rows
INTF675: CXM DPB Schnittstelle -> 675 CXM DPB Schnittstelle with 122 rows
INTF1112: CXM ProductUp FR -> 1112 CXM ProductUp FR with 73 rows
INTF2187: Chausson_Tarifs SIKA -> 2187 Chausson_Tarifs SIKA with 1 rows
INTF5420: Excel ADDOK_MATCO:01.02.2022 Tabelle: Correspondance Nom douaniere -> 5420 Excel ADDOK_MATCO_01.02.2 with 1 rows
INTF5424: Excel ADDOK_MATCO:01.02.2022 Tabelle: Matrice Tabelle: Examples -> 5424 Excel ADDOK_MATCO_01.02.2 with 1 rows
INTF5423: Excel ADDOK_MATCO:01.02.2022 Tabelle: Notice format de matr

## Summary sheet

In [43]:
summary.write(0, 0, "Summary", title_format)

row = 2
summary.write(row, 0, 'Key', column_head_format)
summary.write(row, 1, 'Name', column_head_format)
summary.write(row, 2, 'Mapped', column_head_format)
summary.write(row, 3, 'Total', column_head_format)
summary.write(row, 4, 'Tables', column_head_format)
summary.write(row, 5, 'Sheet Link', column_head_format)

row = 3
for skey, system in spod['systems'].items():
    summary.write(row, 0, skey)
    summary.write(row, 1, system['name'])
    
    columns = list(filter(lambda c: c['interface-id+'] == skey, spod['columns'].values()))
    mapped = list(filter(lambda c: len(c['attributesmapped']) > 0, columns))
    summary.write(row, 2, len(mapped))
    summary.write(row, 3, len(columns))
    
    summary.write(row, 4, len(system['tables+']))
    
    _, ws = next(iter(filter(lambda t: skey[4:] in t[0], worksheets.items())))
    summary.write_url(row, 5, f"internal:'{ws.get_name()}'!A1", string=f'Sheet {skey[4:]}')
    
    row += 1

## Styling

In [44]:
summary.set_column(0, 0, 20)
summary.set_column(1, 1, 60)
summary.set_column(2, 5, 15)

0

## Write Excel file

In [45]:
version_file = Path(LIBRARY, 'versons.json')
if version_file.is_file():
    with open(version_file, 'r') as src:
        version = json.load(src)
else:
    version = { 'TOOLVERSION': '?.?' }

In [46]:
workbook.set_properties({
    'title':    f"{spod['model']['name']}",
    'subject':  'mapping',
    'author':   f"Excel Mapping Publisher {version['TOOLVERSION']}",
#    'manager':  'D',
#    'company':  'of Wolves',
    'category': 'export',
    'keywords': 'Information Model, Data Models',
    'comments': f"generated with {version['TOOLVERSION']} from model {spod['_imprint_']['Modelversion']}",
    'status':   'Draft',
    'revision': spod['_imprint_'].get('git')
})

In [47]:
workbook.close()
print(f"Wrote {xlsx_destination}")

Wrote IM_mapping.xlsx


# Visually verify

In [48]:
import subprocess
r = subprocess.run(['qlmanage', '-x', '-p', xlsx_destination], shell=False) # capture_output=False, stderr=subprocess.DEVNULL)

Testing Quick Look preview with files using server:
	IM_mapping.xlsx


2022-05-27 11:29:33.176 qlmanage[2036:6827692] *** CFMessagePort: bootstrap_register(): failed 1100 (0x44c) 'Permission denied', port = 0xc403, name = 'com.apple.coredrag'
See /usr/include/servers/bootstrap_defs.h for the error codes.
2022-05-27 11:29:33.224 qlmanage[2036:6827692] *** CFMessagePort: bootstrap_register(): failed 1100 (0x44c) 'Permission denied', port = 0xd207, name = 'com.apple.tsm.portname'
See /usr/include/servers/bootstrap_defs.h for the error codes.
2022-05-27 11:29:33.347 qlmanage[2036:6827723] NetworkStorageDB:_openDBReadConnections: failed to open read connection to DB @ /Users/bue/Library/Caches/com.apple.quicklook.qlmanage/Cache.db.  Error=14. Cause=unable to open database file
2022-05-27 11:29:33.347 qlmanage[2036:6827723] The read-connection to the DB=/Users/bue/Library/Caches/com.apple.quicklook.qlmanage/Cache.db is NOT valid.  Unable to determine schema version.
2022-05-27 11:29:33.347 qlmanage[2036:6827723] NetworkStorageDB:_openDBWriteConnections: failed 

In [49]:
import pandas
excel_data_df = pandas.read_excel(DESTINATION, sheet_name='Mapping')

In [50]:
from IPython.display import display, HTML
display(excel_data_df)

,FQN,EID,AID,Entity (EN),Attribute (EN),Entity (DE),Attribute (DE),Entity (FR),Attribute (FR),Examples,...,SAP Export - EAN doublon et code sap remplacant,SAP Export - fichier unique données sap,SAP-ERP,Sika standard,Standart Abmessungen,Standart KD,Standart Link,VFG,Wertschöpfer,free
0,ENTI107:ATTR108,ENTI107,ATTR108,Customer Article,Number,Kunde Artikel,Nummer,Article du client,Numéro,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ENTI107:ATTR109,ENTI107,ATTR109,Customer Article,Designation,Kunde Artikel,Bezeichnung,Article du client,Désignation,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ENTI110:ATTR111,ENTI110,ATTR111,Special provision,No,Sondervorschrift,Nr,Disposition spéciale,N°,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ENTI110:ATTR112,ENTI110,ATTR112,Special provision,Designation,Sondervorschrift,Bezeichnung,Disposition spéciale,Désignation,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ENTI113:ATTR115,ENTI113,ATTR115,Currency,Name,Währung,Name,Monnaie,Nom,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
890,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
891,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,lagkl,NaN,NaN,NaN,NaN,NaN,NaN,NaN
892,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
893,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
